In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (classification_report, roc_auc_score,
                              f1_score, recall_score, confusion_matrix)

In [2]:
data = pd.read_csv("../data/clean/build_dataset.csv")

In [3]:
#data.info()

In [4]:
#data.columns

In [5]:
X = data.drop("churn_value", axis=1)
y = data["churn_value"]

In [6]:
# stratify -> The churn rate (26%) is maintained in both sets; if stratify is not used, it may randomly split unevenly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
print(f"Train : {X_train.shape[0]} rows | Test : {X_test.shape[0]} rows")
print(f"Train churn rate : {y_train.mean():.2%}")
print(f"Test  churn rate : {y_test.mean():.2%}\n")

Train : 5634 rows | Test : 1409 rows
Train churn rate : 26.54%
Test  churn rate : 26.54%



In [8]:
numeric_cols = [
    "tenure_months", "monthly_charges", "total_charges", "cltv", "avg_monthly_charge", "charge_vs_avg", "payment_ratio",
    "total_services", "cost_per_service", "high_risk", "engagement_score", "log_total_charges", "log_monthly_charges",
    "log_cltv", "tenure_x_contract", "charge_x_risk",
]

In [9]:
binary_cols = [
    "senior_citizen", "partner", "dependents", "phone_service", "paperless_billing", "online_security", "online_backup",
    "device_protection", "tech_support", "streaming_tv", "streaming_movies",
]

In [10]:
gender_cols = ["gender"]

In [11]:
ordinal_cols        = ["contract", "tenure_group"]
contract_categories = [["Month-to-month", "One year", "Two year"]]
tenure_categories   = [["New", "Growing", "Mature", "Loyal"]]

In [12]:
nominal_cols = ["multiple_lines", "internet_service", "payment_method"]

In [13]:
preprocessor = ColumnTransformer(transformers=[
    ("num",     StandardScaler(),
                numeric_cols),

    ("binary",  OrdinalEncoder(categories=[["No", "Yes"]] * len(binary_cols)),
                binary_cols),

    ("gender",  OrdinalEncoder(categories=[["Female", "Male"]]),
                gender_cols),

    ("ordinal", OrdinalEncoder(categories=contract_categories + tenure_categories),
                ordinal_cols),

    ("nominal", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
                nominal_cols),
],
remainder="drop"
)

In [14]:
X_train_encoded = preprocessor.fit_transform(X_train, y_train)
X_test_encoded = preprocessor.transform(X_test)

In [15]:
X_train = pd.DataFrame(X_train_encoded, columns=preprocessor.get_feature_names_out(), index=X_train.index)
X_test = pd.DataFrame(X_test_encoded, columns=preprocessor.get_feature_names_out(), index=X_test.index)

In [16]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 4626 to 6017
Data columns (total 36 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   num__tenure_months                               5634 non-null   float64
 1   num__monthly_charges                             5634 non-null   float64
 2   num__total_charges                               5634 non-null   float64
 3   num__cltv                                        5634 non-null   float64
 4   num__avg_monthly_charge                          5634 non-null   float64
 5   num__charge_vs_avg                               5634 non-null   float64
 6   num__payment_ratio                               5634 non-null   float64
 7   num__total_services                              5634 non-null   float64
 8   num__cost_per_service                            5634 non-null   float64
 9   num__high_risk                             

In [17]:
def calculate_classification_metrics(y_true, y_pred, y_prob):
    f1 = f1_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)

    return f1, recall, roc_auc, cm

In [18]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

classification_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, verbosity=0, n_jobs=-1),
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1)
}

In [19]:
model_names = list(classification_models.keys())
results = {}

for name in model_names:
    model = classification_models[name]

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Sınıf Olasılık Tahminleri (ROC AUC skoru için 1 olma olasılığı gerekir)
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    train_f1, train_recall, train_roc_auc, train_cm = calculate_classification_metrics(
        y_train, y_train_pred, y_train_prob
    )
    test_f1, test_recall, test_roc_auc, test_cm = calculate_classification_metrics(
        y_test, y_test_pred, y_test_prob
    )

    print(f"=== {name} ===")

    print("Evaluation for Training Set")
    print("F1 Score  :", round(train_f1, 4))
    print("Recall    :", round(train_recall, 4))
    print("ROC AUC   :", round(train_roc_auc, 4))
    print("Confusion Matrix :\n", train_cm)

    print("------------------------")

    print("Evaluation for Test Set")
    print("F1 Score  :", round(test_f1, 4))
    print("Recall    :", round(test_recall, 4))
    print("ROC AUC   :", round(test_roc_auc, 4))
    print("Confusion Matrix :\n", test_cm)

    print("------------------------")
    print(classification_report(y_test, y_test_pred))

    print("\n")

    results[name] = {
        "Train ROC-AUC": round(train_roc_auc, 4),
        "Test ROC-AUC" : round(test_roc_auc, 4),
        "Test F1"      : round(test_f1, 4),
        "Test Recall"  : round(test_recall, 4),
        "Overfit"      : round(train_roc_auc - test_roc_auc, 4),
    }

=== Logistic Regression ===
Evaluation for Training Set
F1 Score  : 0.6547
Recall    : 0.8268
ROC AUC   : 0.8669
Confusion Matrix :
 [[3094 1045]
 [ 259 1236]]
------------------------
Evaluation for Test Set
F1 Score  : 0.6216
Recall    : 0.7861
ROC AUC   : 0.8556
Confusion Matrix :
 [[757 278]
 [ 80 294]]
------------------------
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



=== Random Forest ===
Evaluation for Training Set
F1 Score  : 1.0
Recall    : 1.0
ROC AUC   : 1.0
Confusion Matrix :
 [[4139    0]
 [   0 1495]]
------------------------
Evaluation for Test Set
F1 Score  : 0.5477
Recall    : 0.4759
ROC AUC   : 0.8437
Confusion Matrix :
 [[937  98]
 [196 178]]
------------------------
              prec

In [20]:
results_df = pd.DataFrame(results).T.sort_values("Test ROC-AUC", ascending=False)
print(results_df)

                     Train ROC-AUC  Test ROC-AUC  Test F1  Test Recall  \
Logistic Regression         0.8669        0.8556   0.6216       0.7861   
LightGBM                    0.9770        0.8488   0.6237       0.7246   
Random Forest               1.0000        0.8437   0.5477       0.4759   
XGBoost                     0.9984        0.8345   0.6025       0.6444   

                     Overfit  
Logistic Regression   0.0113  
LightGBM              0.1282  
Random Forest         0.1563  
XGBoost               0.1639  
